# SnowHealth Demand Forecasting — Model Monitoring

Sets up **Snowflake Model Version Monitoring** to detect drift in the demand forecasting model's predictions and input features over time.

| Step | What it does |
|---|---|
| **Setup** | Load model, verify predictions table from notebook 05 |
| **Baseline** | Snapshot current predictions as the baseline distribution |
| **Simulate Drift** | Insert rows with shifted feature distributions (temperature, inventory) |
| **Create Monitor** | `CREATE MODEL MONITOR` with drift detection (regression task) |
| **Query Metrics** | PSI, Difference of Means, Wasserstein distance on key features |

> **Prerequisite:** Run `05_inference.ipynb` through the "Materialize Predictions" cell to create the predictions table.

### Initialize Session and Verify Predictions Table
Connects to Snowflake, loads the model registry reference, and verifies the predictions table created by notebook 05 exists with the expected schema (features + prediction + timestamp).

In [ ]:
import sys, os

# Resolve source/ (works from root or notebooks/ subdirectory)
for _p in [os.path.join(os.getcwd(), 'source'), os.path.join(os.getcwd(), '..', 'source')]:
    if os.path.isdir(_p): sys.path.insert(0, os.path.realpath(_p)); break

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
import config as cfg
import pandas as pd
import numpy as np

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()

from snowflake.ml.registry import Registry
registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
default_version = model_ref.default
print(f"Model: {cfg.SNOWHEALTH_MODEL_NAME}, version: {default_version.version_name}")

# Create predictions table if it doesn't exist (normally created in notebook 05)
try:
    pred_count = session.table(cfg.SNOWHEALTH_PREDICTIONS_TABLE).count()
except:
    print("Predictions table not found — creating from features table...")
    pred_df = session.table(cfg.SNOWHEALTH_FEATURE_TABLE)
    pred_result = default_version.run(pred_df, function_name="predict")
    pred_result_sf = pred_result.with_column(
        'PREDICTION_TIMESTAMP',
        F.current_timestamp().cast(T.TimestampType(T.TimestampTimeZone.NTZ))
    )
    pred_result_sf.write.mode('overwrite').save_as_table(cfg.SNOWHEALTH_PREDICTIONS_TABLE)
    pred_count = session.table(cfg.SNOWHEALTH_PREDICTIONS_TABLE).count()

cols = session.table(cfg.SNOWHEALTH_PREDICTIONS_TABLE).columns
print(f"Predictions table: {cfg.SNOWHEALTH_PREDICTIONS_TABLE} ({pred_count:,} rows)")
print(f"Prediction column: UNITS_SOLD (model output for regression target)")
print(f"Timestamp column: PREDICTION_TIMESTAMP")

### Create Baseline Table
Snapshots the current predictions as the **baseline distribution**. The model monitor compares future predictions against this baseline to detect drift. This represents the "known good" state of the model at deployment time.

In [ ]:
# Create baseline table — snapshot of current predictions
baseline_table = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_MONITOR_BASELINE"

session.table(cfg.SNOWHEALTH_PREDICTIONS_TABLE).write.mode('overwrite').save_as_table(baseline_table)
print(f"Baseline table created: {baseline_table}")
print(f"Rows: {session.table(baseline_table).count():,}")

### Simulate Feature Drift
To demonstrate drift detection, we insert new rows with **shifted feature distributions** into the predictions table. This simulates a real-world scenario where input data characteristics change over time (e.g., a heat wave raises temperatures, a supply chain disruption alters inventory levels).

**Drift injected:**
- `TEMPERATURE_F` shifted +15°F (simulates unseasonable heat)
- `INVENTORY_ON_HAND` reduced by 40% (simulates supply shortage)
- `FLU_INDEX` doubled (simulates flu outbreak)
- Predictions re-scored with the drifted features, timestamped 7 days in the future

In [ ]:
from snowflake.ml.registry import Registry
from feature_engineering import get_feature_columns
import datetime

registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
default_version = model_ref.default

# Load the current predictions and apply drift to key features
drift_pdf = session.table(cfg.SNOWHEALTH_PREDICTIONS_TABLE).to_pandas()
original_temp = drift_pdf['TEMPERATURE_F'].mean()
original_inv = drift_pdf['INVENTORY_ON_HAND'].mean()
original_flu = drift_pdf['FLU_INDEX'].mean()

drift_pdf['TEMPERATURE_F'] = drift_pdf['TEMPERATURE_F'] + 15.0
drift_pdf['INVENTORY_ON_HAND'] = (drift_pdf['INVENTORY_ON_HAND'] * 0.6).astype(int)
drift_pdf['FLU_INDEX'] = drift_pdf['FLU_INDEX'] * 2.0

print(f"Drift applied to {len(drift_pdf):,} rows")
print(f"  TEMPERATURE_F: mean {original_temp:.1f} → {drift_pdf['TEMPERATURE_F'].mean():.1f}")
print(f"  INVENTORY_ON_HAND: mean {original_inv:.0f} → {drift_pdf['INVENTORY_ON_HAND'].mean():.0f}")
print(f"  FLU_INDEX: mean {original_flu:.2f} → {drift_pdf['FLU_INDEX'].mean():.2f}")

# Re-score through the model with drifted features
feature_cols = get_feature_columns(drift_pdf, cfg.TARGET_COLUMN)
drift_snow = session.create_dataframe(drift_pdf[feature_cols].reset_index(drop=True))
drift_preds = default_version.run(drift_snow, function_name="predict").to_pandas()
drift_pdf['UNITS_SOLD'] = drift_preds['UNITS_SOLD'].values

# Timestamp 7 days in the future to separate from baseline
drift_pdf['PREDICTION_TIMESTAMP'] = pd.Timestamp.now() + datetime.timedelta(days=7)

# Append to predictions table
session.create_dataframe(drift_pdf.reset_index(drop=True)).write.mode('append').save_as_table(cfg.SNOWHEALTH_PREDICTIONS_TABLE)
total = session.table(cfg.SNOWHEALTH_PREDICTIONS_TABLE).count()
print(f"\nAppended {len(drift_pdf):,} drifted rows")
print(f"Total rows: {total:,} (baseline + drifted)")

### Create Model Monitor
Creates a **Model Version Monitor** that tracks drift between the baseline distribution and incoming predictions. For a regression task (demand forecasting), the monitor tracks:

- **Drift metrics**: Population Stability Index (PSI), Difference of Means, Wasserstein distance
- **Statistical metrics**: count, nulls, min/max/avg on prediction and feature columns

The monitor refreshes every 60 seconds and aggregates over a 1-day window.

In [ ]:
%%sql -r drop_monitor_result
DROP MODEL MONITOR IF EXISTS SNOWHEALTH_DEMAND_MONITOR

In [ ]:
CREATE MODEL MONITOR SNOWHEALTH_DEMAND_MONITOR WITH
    MODEL = SNOWHEALTH_DEMAND_FORECAST
    VERSION = 'TUNED_V1'
    FUNCTION = 'PREDICT'
    SOURCE = DEV_SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS.SNOWHEALTH_DEMAND_PREDICTIONS
    WAREHOUSE = COMPUTE_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('UNITS_SOLD')
    BASELINE = DEV_SNOWHEALTH_ML_DB.SNOWHEALTH_MODELS.SNOWHEALTH_MONITOR_BASELINE

### Verify Monitor Status
Checks that the monitor was created successfully and is in `ACTIVE` state. The first refresh may take a minute to populate drift metrics.

In [ ]:
%%sql -r monitor_desc
DESC MODEL MONITOR SNOWHEALTH_DEMAND_MONITOR

In [ ]:
import pandas as pd
monitor_pdf = monitor_desc.to_pandas() if not isinstance(monitor_desc, pd.DataFrame) else monitor_desc
print(f"Monitor state: {monitor_pdf['monitor_state'].iloc[0]}")
print(f"Aggregation status: {monitor_pdf['aggregation_status'].iloc[0]}")
if 'aggregation_last_error' in monitor_pdf.columns:
    err = monitor_pdf['aggregation_last_error'].iloc[0]
    if err and str(err).strip():
        print(f"Last error: {err}")

### Query Drift Metrics
Queries the monitor for drift on key features. For a regression/forecasting model, the most relevant drift metrics are:

| Metric | What it measures | Alert threshold |
|---|---|---|
| **Population Stability Index (PSI)** | Distribution shift between baseline and current | >0.2 = significant drift |
| **Difference of Means** | Shift in average feature values | Context-dependent |
| **Wasserstein Distance** | Earth-mover distance between distributions | Higher = more drift |

In [ ]:
%%sql -r psi_predictions
-- PSI on the prediction column (UNITS_SOLD)
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'SNOWHEALTH_DEMAND_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'UNITS_SOLD',
    '1 DAY',
    DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    DATEADD('day', 30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ
))

In [ ]:
%%sql -r dom_temperature
-- Difference of Means on TEMPERATURE_F (we shifted it +15°F)
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'SNOWHEALTH_DEMAND_MONITOR',
    'DIFFERENCE_OF_MEANS',
    'TEMPERATURE_F',
    '1 DAY',
    DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    DATEADD('day', 30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ
))

In [ ]:
%%sql -r wass_inventory
-- Wasserstein distance on INVENTORY_ON_HAND (we reduced it by 40%)
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'SNOWHEALTH_DEMAND_MONITOR',
    'WASSERSTEIN',
    'INVENTORY_ON_HAND',
    '1 DAY',
    DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    DATEADD('day', 30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ
))

In [ ]:
%%sql -r psi_flu
-- PSI on FLU_INDEX (we doubled it)
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'SNOWHEALTH_DEMAND_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'FLU_INDEX',
    '1 DAY',
    DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    DATEADD('day', 30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ
))

### Drift Analysis Summary
Collects all drift metrics into a single view showing which features have drifted and by how much. Features with PSI > 0.2 or large Wasserstein distances indicate the model may need retraining.

In [ ]:
import pandas as pd

def safe_to_pandas(df):
    return df.copy() if isinstance(df, pd.DataFrame) else df.to_pandas()

def get_max_drift(result_var):
    pdf = safe_to_pandas(result_var)
    if len(pdf) > 0 and 'METRIC_VALUE' in pdf.columns:
        return pdf['METRIC_VALUE'].max()
    return None

print('=' * 60)
print('  DRIFT DETECTION RESULTS')
print('=' * 60)

for label, result_var in [
    ('UNITS_SOLD prediction (PSI)', psi_predictions),
    ('TEMPERATURE_F (Diff of Means)', dom_temperature),
    ('INVENTORY_ON_HAND (Wasserstein)', wass_inventory),
    ('FLU_INDEX (PSI)', psi_flu),
]:
    val = get_max_drift(result_var)
    if val is not None:
        flag = ' *** DRIFT DETECTED' if val > 0.2 else ''
        print(f"  {label:45s} = {val:.4f}{flag}")
    else:
        print(f"  {label:45s} = (awaiting first refresh)")

print('=' * 60)
print('  PSI > 0.2 indicates significant distribution shift')
print('  Action: investigate root cause, consider retraining')
print('=' * 60)

### Summary
Recap of the model monitoring pipeline.

In [ ]:
print('=' * 60)
print('  MODEL MONITORING NOTEBOOK COMPLETE')
print('=' * 60)
print(f'  Model:          {cfg.SNOWHEALTH_MODEL_NAME}')
print(f'  Version:        TUNED_V1')
print(f'  Monitor:        SNOWHEALTH_DEMAND_MONITOR')
print(f'  Source table:   {cfg.SNOWHEALTH_PREDICTIONS_TABLE}')
print(f'  Baseline:       SNOWHEALTH_MONITOR_BASELINE')
print(f'  Drift injected: TEMPERATURE_F +15°F, INVENTORY -40%, FLU_INDEX x2')
print(f'')
print(f'  Drift metrics tracked:')
print(f'    - Population Stability Index (PSI)')
print(f'    - Difference of Means')
print(f'    - Wasserstein Distance')
print(f'')
print(f'  View in Snowsight: AI&ML → Models → {cfg.SNOWHEALTH_MODEL_NAME} → Monitors')
print('=' * 60)